# A/B Testing Bayesiano em Farmacovigilância: Semaglutida vs Tirzepatida

**Autor:** Luanda Rodrigues | **Papel:** Analista de Dados Sênior  
**Tags:** Teste A/B Bayesiano, Healthcare Analytics, Farmacovigilância, OpenFDA, Python

---

## 1. O Problema de Negócio: Muito Além da Média

No mundo real das operações de saúde e farmacovigilância, decisões baseadas apenas em médias ou totais de um dashboard podem esconder diferenças relevantes na gravidade dos relatos.

A expansão dos medicamentos agonistas de GLP-1/GIP, como **Ozempic/Wegovy** e **Mounjaro/Zepbound**, mudou o cenário farmacêutico. Uma pergunta útil para gestão de risco é: **entre os eventos adversos notificados à FDA desde janeiro de 2024, existe diferença na proporção de relatos classificados como graves entre semaglutida e tirzepatida?**

Este notebook usa dados públicos do **openFDA FAERS** e aplica um modelo **Beta-Binomial Bayesiano** para transformar incerteza em uma probabilidade direta: *qual é a chance de a proporção de relatos graves ser maior para uma droga do que para a outra?*

> Nota metodológica: FAERS/openFDA reúne notificações espontâneas. Esses dados são úteis para sinalização e priorização de investigação, mas não estimam incidência populacional, não controlam exposição e não provam causalidade clínica.

In [ ]:
# Configuração de ambiente e bibliotecas
from datetime import date
import warnings

import requests
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

BASE_URL = 'https://api.fda.gov/drug/event.json'
START_DATE = '20240101'
END_DATE = date.today().strftime('%Y%m%d')
DRUGS = {
    'SEMAGLUTIDE': 'Semaglutida',
    'TIRZEPATIDE': 'Tirzepatida',
}
COLORS = {
    'Semaglutida': '#0f766e',
    'Tirzepatida': '#7c3aed',
    'Accent': '#f59e0b',
}
RNG = np.random.default_rng(42)

print(f'Janela de coleta: {START_DATE} a {END_DATE}')

## 2. Ingestão de Dados: OpenFDA na Prática

Vamos conectar diretamente na API de eventos adversos da FDA e extrair duas visões:

1. **Série temporal:** crescimento dos relatos ao longo dos meses.
2. **Proporção de gravidade:** quantos relatos foram classificados como graves (`serious = 1`) versus não graves (`serious = 2`).

O notebook inclui tratamento básico de erro para deixar a execução reproduzível em Colab/Jupyter e falhar com mensagens claras quando a API muda, limita acesso ou não retorna dados.

In [ ]:
def make_search_query(drug_name):
    return f'patient.drug.medicinalproduct:"{drug_name}" AND receiptdate:[{START_DATE} TO {END_DATE}]'


def fda_count(drug_name, count_field):
    params = {
        'search': make_search_query(drug_name),
        'count': count_field,
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    if response.status_code == 404:
        return [], response.json().get('meta', {})
    response.raise_for_status()
    payload = response.json()
    return payload.get('results', []), payload.get('meta', {})


def fetch_fda_timeseries(drug_name):
    results, meta = fda_count(drug_name, 'receiptdate')
    if not results:
        return pd.DataFrame(columns=['time', 'count', 'drug', 'cumulative_count']), meta

    df = pd.DataFrame(results)
    df['time'] = pd.to_datetime(df['time'], format='%Y%m%d')
    df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)
    df = (
        df.set_index('time')[['count']]
        .resample('MS')
        .sum()
        .reset_index()
    )
    df['drug'] = DRUGS[drug_name]
    df['cumulative_count'] = df['count'].cumsum()
    return df, meta


frames = []
metadata = {}
for drug in DRUGS:
    df_drug, meta = fetch_fda_timeseries(drug)
    frames.append(df_drug)
    metadata[drug] = meta

df_timeseries = pd.concat(frames, ignore_index=True)
if df_timeseries.empty:
    raise ValueError('A API openFDA nao retornou dados para a janela selecionada.')

df_timeseries['month_str'] = df_timeseries['time'].dt.strftime('%Y-%m')
last_updated = next((m.get('last_updated') for m in metadata.values() if m.get('last_updated')), 'nao informado')
print(f'Ultima atualizacao informada pela openFDA: {last_updated}')
df_timeseries.tail()

## 3. Análise Exploratória (EDA): O Crescimento dos Relatos

A visualização abaixo mostra a evolução acumulada dos relatos desde janeiro de 2024. Como o volume bruto depende de exposição, tempo de mercado, comportamento de notificação e cobertura midiática, ele deve ser lido como **atividade de notificação**, não como risco absoluto do medicamento.

In [ ]:
# Grafico animado da evolucao de relatos
max_y = max(1, df_timeseries['cumulative_count'].max()) * 1.1
fig = px.bar(
    df_timeseries,
    x='drug',
    y='cumulative_count',
    color='drug',
    animation_frame='month_str',
    animation_group='drug',
    range_y=[0, max_y],
    color_discrete_map=COLORS,
    title='Crescimento Acumulado de Notificacoes a FDA desde Jan/2024',
    labels={
        'cumulative_count': 'Notificacoes acumuladas',
        'drug': 'Medicamento',
        'month_str': 'Mes',
    },
)

fig.update_layout(
    template='plotly_white',
    title_font_size=20,
    font=dict(family='Arial, sans-serif'),
    showlegend=False,
)

if fig.layout.updatemenus:
    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 400

fig.show()

## 4. O Teste A/B Bayesiano: Proporção de Relatos Graves

O volume absoluto não conta a história toda. Para uma comparação mais útil, vamos modelar a **proporção de notificações classificadas como graves** entre todos os relatos com `serious` preenchido.

Mesmo assim, esta comparação continua sendo observacional e sujeita a vieses de notificação. O resultado deve ser interpretado como um sinal de farmacovigilância a investigar, não como conclusão causal de segurança clínica.

In [ ]:
def fetch_fda_seriousness(drug_name):
    results, meta = fda_count(drug_name, 'serious')
    counts = {item.get('term'): int(item.get('count', 0)) for item in results}
    serious = counts.get(1, 0)
    non_serious = counts.get(2, 0)
    return serious, non_serious, meta


summary_rows = []
for drug, label in DRUGS.items():
    serious, non_serious, meta = fetch_fda_seriousness(drug)
    total = serious + non_serious
    if total == 0:
        raise ValueError(f'Nenhum relato com campo serious foi encontrado para {label}.')
    summary_rows.append({
        'drug_code': drug,
        'Medicamento': label,
        'Graves': serious,
        'Nao graves': non_serious,
        'Total': total,
        'Taxa observada': serious / total,
        'openFDA last_updated': meta.get('last_updated', 'nao informado'),
    })

seriousness_df = pd.DataFrame(summary_rows)
display_df = seriousness_df.copy()
display_df['Taxa observada'] = display_df['Taxa observada'].map(lambda value: f'{value:.1%}')
display_df.drop(columns=['drug_code'])

### 4.1. Modelagem Bayesiana (Distribuição Beta-Binomial)

A estatística Bayesiana trata a taxa real de gravidade como uma distribuição de probabilidade. Usaremos um prior não informativo `Beta(1, 1)` e atualizaremos com os dados observados:

`Posterior = Beta(alpha_prior + graves, beta_prior + total - graves)`

Além da média posterior, vamos calcular um intervalo credível de 95% para mostrar a incerteza restante.

In [ ]:
alpha_prior = 1
beta_prior = 1
posteriors = {}
posterior_rows = []

for _, row in seriousness_df.iterrows():
    posterior = stats.beta(alpha_prior + row['Graves'], beta_prior + row['Total'] - row['Graves'])
    posteriors[row['Medicamento']] = posterior
    ci_low, ci_high = posterior.ppf([0.025, 0.975])
    posterior_rows.append({
        'Medicamento': row['Medicamento'],
        'Taxa observada': row['Taxa observada'],
        'Media posterior': posterior.mean(),
        'ICr 95% inferior': ci_low,
        'ICr 95% superior': ci_high,
    })

posterior_summary = pd.DataFrame(posterior_rows)
posterior_summary_display = posterior_summary.copy()
for col in ['Taxa observada', 'Media posterior', 'ICr 95% inferior', 'ICr 95% superior']:
    posterior_summary_display[col] = posterior_summary_display[col].map(lambda value: f'{value:.2%}')
posterior_summary_display

### 4.2. Visualização das Distribuições Posteriores

Quando as curvas ficam bem separadas, há pouca sobreposição entre as taxas plausíveis para cada medicamento dentro desta amostra de notificações.

In [ ]:
low = max(0, posterior_summary['ICr 95% inferior'].min() - 0.05)
high = min(1, posterior_summary['ICr 95% superior'].max() + 0.05)
x = np.linspace(low, high, 1000)

fig = go.Figure()
for label, posterior in posteriors.items():
    fig.add_trace(go.Scatter(
        x=x,
        y=posterior.pdf(x),
        mode='lines',
        fill='tozeroy',
        name=label,
        marker_color=COLORS[label],
        opacity=0.72,
    ))

fig.update_layout(
    title='Distribuicao Posterior da Taxa de Relatos Graves',
    xaxis_title='Proporcao verdadeira de relatos graves',
    yaxis_title='Densidade de probabilidade',
    template='plotly_white',
    font=dict(family='Arial, sans-serif'),
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

## 5. Decisão Executiva: Qual é a Chance Real?

Agora calculamos a probabilidade direta de a taxa de relatos graves da Semaglutida ser maior do que a da Tirzepatida, além da diferença esperada entre as duas proporções.

In [ ]:
n_simulations = 100_000
sim_sema = posteriors['Semaglutida'].rvs(n_simulations, random_state=RNG)
sim_tirz = posteriors['Tirzepatida'].rvs(n_simulations, random_state=RNG)

diff = sim_sema - sim_tirz
prob_sema_maior = np.mean(diff > 0)
diff_low, diff_high = np.quantile(diff, [0.025, 0.975])

print(
    'Probabilidade Bayesiana de a Semaglutida ter maior proporcao de relatos graves: '
    f'{prob_sema_maior:.4%}'
)
print(f'Diferenca media posterior: {diff.mean():.2%} pontos percentuais')
print(f'Intervalo credivel 95% da diferenca: [{diff_low:.2%}, {diff_high:.2%}]')

## 6. Veredito e Valor para o Negócio

Dentro dos relatos espontâneos disponíveis no openFDA para a janela analisada, o modelo Bayesiano estima a probabilidade de a Semaglutida apresentar maior proporção de relatos graves do que a Tirzepatida. A leitura executiva correta é: **há um sinal forte nos dados de notificação que merece monitoramento e investigação**, especialmente em protocolos de farmacovigilância, auditoria de qualidade e priorização de revisão clínica.

### O que isso significa para a operação de saúde?

- **Gestão de risco:** usar o sinal para priorizar revisão de eventos graves, não para concluir causalidade isoladamente.
- **Monitoramento clínico:** acompanhar a evolução mensal e repetir a análise quando a openFDA atualizar a base.
- **Comunicação executiva:** reportar probabilidades e intervalos credíveis torna a incerteza mais transparente do que uma decisão binária baseada só em p-valor.

**Limitações importantes:** FAERS/openFDA não contém denominador de pacientes expostos, é sensível a subnotificação, notoriedade pública, duplicidade e severidade percebida. Portanto, o notebook apoia sinalização analítica e tomada de decisão exploratória, não substitui estudos epidemiológicos controlados.